In [1]:
import os
import time
import re
import pandas as pd
from datetime import datetime, timedelta
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

In [2]:
# ==========================================
# 📌 함수 1. 기본 정보 추출
# ==========================================
def get_kream_basic_info(product_id):
    url = f"https://kream.co.kr/products/{product_id}"
    if driver.current_url != url: driver.get(url)
    
    time.sleep(1.5) 
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    info = {"한글명": "Unknown", "관심수": 0}
    
    target_selector = f'[data-sdui-id="product_wish_count/{product_id}"]'
    wish_element = soup.select_one(target_selector)
    if not wish_element: wish_element = soup.select_one('[data-sdui-id*="product_wish_count"]')
    
    if wish_element:
        wish_str = wish_element.get_text(strip=True)
        if '만' in wish_str:
            num_str = re.sub(r'[^0-9.]', '', wish_str)
            info["관심수"] = int(float(num_str) * 10000) if num_str else 0
        else:
            num_str = re.sub(r'[^0-9]', '', wish_str)
            info["관심수"] = int(num_str) if num_str else 0

    p_tags = soup.find_all('p')
    for p in p_tags:
        style = p.get('style', '')
        if 'font-size:15' in style and 'line-clamp:1' in style:
            info["한글명"] = p.get_text(strip=True)
            break
    return info

In [3]:
# ==========================================
# 🌟 함수 2. 상세 정보 추출 (모델번호 포함)
# ==========================================
def get_kream_details(product_id):
    print(f"\n🚨 [{product_id}] '혜택 더보기' 위의 '>' 버튼을 눌러 상세 정보를 펴주세요.")
    input("👉 모델번호/발매일/색상 등이 보이면 엔터(Enter)를 치세요! : ")
    
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    details = {"모델번호": None, "발매일": None, "발매가": None, "색상": None} 
    
    p_tags = soup.find_all('p')
    for p in p_tags:
        text = p.get_text(strip=True)
        
        if text.startswith("모델번호"):
            parts = text.split(maxsplit=1)
            if len(parts) > 1: 
                val = parts[1].strip()
                if "정보 없음" not in val and val != "-":
                    details["모델번호"] = val

        elif text.startswith("발매일"):
            parts = text.split(maxsplit=1)
            if len(parts) > 1: 
                val = parts[1].strip()
                match = re.search(r'\d{2}/\d{2}/\d{2}', val)
                if match:
                    details["발매일"] = match.group()
                    
        elif text.startswith("발매가"):
            parts = text.split(maxsplit=1)
            if len(parts) > 1:
                val = parts[1].strip()
                price_str = re.sub(r'[^0-9]', '', val) 
                details["발매가"] = int(price_str) if price_str else None
                
        elif text.startswith("색상"):
            parts = text.split(maxsplit=1)
            if len(parts) > 1:
                val = parts[1].strip()
                if "정보 없음" not in val and val != "-":
                    details["색상"] = val
                    
    return details

In [4]:
# ==========================================
# 🌟 함수 3. 거래 내역 추출 (최대 2,000개 수집 및 사이즈 타겟팅)
# ==========================================
def get_kream_transactions(product_id):
    print(f"\n🚨 [{product_id}] '체결 내역' 창을 열어주세요!")
    print(f"💡 정렬 버튼(↑↓)을 눌러 '과거순'이 맨 위로 오게 한 뒤,")
    print(f"   스크롤을 충분히 내려 최대 2,000개 정도의 거래내역이 화면에 로딩되도록 해주세요.")
    input("👉 정렬과 스크롤을 마쳤으면 엔터(Enter)를 치세요! : ")
    
    def parse_kream_date(d_str):
        d_str = str(d_str)
        # "X시간 전", "X분 전"은 현재 시스템의 오늘 날짜로 변환
        if "분 전" in d_str or "시간 전" in d_str: 
            return today_date
        elif "일 전" in d_str:
            days_ago = int(re.sub(r'[^0-9]', '', d_str))
            return today_date - timedelta(days=days_ago)
        else:
            match = re.search(r'\d{2}/\d{2}/\d{2}', d_str)
            if match: 
                return pd.to_datetime("20" + match.group(), format="%Y/%m/%d").date()
            return None

    soup = BeautifulSoup(driver.page_source, 'html.parser')
    rows = soup.find_all('div', class_='body_list')
    
    transactions = []
    
    # 🌟 조건 1: 최대 2000개까지만 추출
    count = 0
    for row in rows:
        if count >= 2000:
            break
            
        cols = row.find_all('div', class_='list_txt')
        if len(cols) >= 3:
            price_str = cols[1].get_text(strip=True)
            if "원" not in price_str: continue
            
            date_str = cols[2].get_text(strip=True)
            trade_date = parse_kream_date(date_str)
            if not trade_date: continue

            size_raw = cols[0].get_text(strip=True)
            match = re.search(r'\d{3}', size_raw) 
            if not match: continue
            size_num = match.group() 
                
            price = int(re.sub(r'[^0-9]', '', price_str))
            
            transactions.append({
                "size": size_num,
                "price": price,
                "date": trade_date
            })
            count += 1
            
    # 전체 긁어온 거래 내역 데이터프레임
    df = pd.DataFrame(transactions)
    scraped_count = len(df)
    
    if scraped_count == 0:
        return {"scraped_count": 0, "golden_avg_price": None}
        
    # 🌟 조건 2: 남녀공용 타겟 사이즈 추출 및 평균 계산
    target_sizes = ['235', '240', '245', '265', '270', '275']
    df_target = df[df['size'].isin(target_sizes)]
    
    if df_target.empty:
        golden_avg = None
    else:
        golden_avg = int(df_target['price'].mean())
        
    return {
        "scraped_count": scraped_count, 
        "golden_avg_price": golden_avg
    }

In [7]:
# 1. 크롬 드라이버 셋팅
print("🌐 크롬 브라우저를 셋팅합니다...")
driver_path = ChromeDriverManager().install()
os.system(f"xattr -cr '{driver_path}'")
os.system("xattr -cr ~/.wdm")

options = webdriver.ChromeOptions()
options.add_argument("user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
options.add_experimental_option("detach", True)
driver = webdriver.Chrome(service=Service(driver_path), options=options)

today_date = datetime.now().date()

# 수집할 상품 ID 리스트 (남녀공용 나이키 신발 위주로 구성)
my_product_ids = [
    138416,
    282500,
    632282,
    465724,
    741721,
    266659,
    557949,
    501037,
    611602,
    416242,
    788722,
    595442,
    21438,
    602680,
    137241,
    611603,
    78768,
    754815,
    526264,
    402232
]

ml_dataset = []

print("🚀 KREAM 남녀공용 타겟 데이터 수집\n" + "="*50)

for pid in my_product_ids:
    # 1. 기본 정보 
    basic_info = get_kream_basic_info(pid)
    
    # 2. 상세 정보
    details_info = get_kream_details(pid)
    
    # 3. 거래 내역 (최대 2000개 수집 및 대상 사이즈 평균)
    tx_data = get_kream_transactions(pid)
        
    # 4. 행(Row)으로 합치기 (최종 CSV 포맷 요구사항 반영)
    row_data = {
        "Product_ID": pid,
        "Name": basic_info["한글명"],
        "Model_No": details_info["모델번호"],
        "Wish_Count": basic_info["관심수"],
        "Release_Date": details_info["발매일"],
        "Retail_Price": details_info["발매가"],
        "Color": details_info.get("색상"),
        "Scraped_Tx_Count": tx_data["scraped_count"],
        "Golden_Avg_Price": tx_data["golden_avg_price"]
    }
    
    ml_dataset.append(row_data)
    print(f"🎉 [{pid}] 수집 및 분석 완료! (긁어온 거래량: {tx_data['scraped_count']}개)\n" + "-"*50)

# 최종 데이터프레임 생성 및 CSV 저장
df_final = pd.DataFrame(ml_dataset)
df_final.to_csv("KREAM_Unisex_TargetSize_Dataset.csv", index=False, encoding="utf-8-sig")

print("\n🎊 모든 데이터셋 생성이 완료되었습니다!")
print("폴더에 'KREAM_Unisex_TargetSize_Dataset.csv' 파일이 생성되었습니다.")
display(df_final)

🌐 크롬 브라우저를 셋팅합니다...
🚀 KREAM 남녀공용 타겟 데이터 수집

🚨 [138416] '혜택 더보기' 위의 '>' 버튼을 눌러 상세 정보를 펴주세요.


👉 모델번호/발매일/색상 등이 보이면 엔터(Enter)를 치세요! :  



🚨 [138416] '체결 내역' 창을 열어주세요!
💡 정렬 버튼(↑↓)을 눌러 '과거순'이 맨 위로 오게 한 뒤,
   스크롤을 충분히 내려 최대 2,000개 정도의 거래내역이 화면에 로딩되도록 해주세요.


👉 정렬과 스크롤을 마쳤으면 엔터(Enter)를 치세요! :  


🎉 [138416] 수집 및 분석 완료! (긁어온 거래량: 2000개)
--------------------------------------------------

🚨 [282500] '혜택 더보기' 위의 '>' 버튼을 눌러 상세 정보를 펴주세요.


...

👉 모델번호/발매일/색상 등이 보이면 엔터(Enter)를 치세요! :  



🚨 [402232] '체결 내역' 창을 열어주세요!
💡 정렬 버튼(↑↓)을 눌러 '과거순'이 맨 위로 오게 한 뒤,
   스크롤을 충분히 내려 최대 2,000개 정도의 거래내역이 화면에 로딩되도록 해주세요.


👉 정렬과 스크롤을 마쳤으면 엔터(Enter)를 치세요! :  


🎉 [402232] 수집 및 분석 완료! (긁어온 거래량: 0개)
--------------------------------------------------

🎊 모든 데이터셋 생성이 완료되었습니다!
폴더에 'KREAM_Unisex_TargetSize_Dataset.csv' 파일이 생성되었습니다.


,Product_ID,Name,Model_No,Wish_Count,Release_Date,Retail_Price,Color,Scraped_Tx_Count,Golden_Avg_Price
0,138416,나이키 줌 보메로 5 SP 블랙,BV1358-003,31000,23/07/03,189000,BLACK/BLACK/WHITE,2000,155143.0
1,282500,나이키 ACG 루퍼스 라임스톤 앤 블랙,FV2923-200,22000,24/05/10,129000,Limestone/Limestone/Black/Black,2000,156953.0
2,632282,나이키 에어포스 1 로우 고어텍스 비브람 오프 누아르 앤 블랙,HV5953-001,5915,25/11/13,209000,Off Noir/Black/Speed Yellow,1667,188198.0
3,465724,나이키 리액트X 리주버네이트 트리플 블랙,HV5060-001,6304,25/01/29,89000,Black/Black/Black,2000,135903.0
4,741721,나이키 x 니고 에어포스 3 로우 미드나잇 네이비 쉐도우 그레이,HV5032-400,242,26/02/06,179000,Midnight Navy/Off White/Shadow Grey,25,197363.0
5,266659,나이키 V2K 런 블랙 앤트러사이트,HJ4497-001,51000,NaN,149000,Black/Anthracite/Dark Smoke Grey,1550,118820.0
6,557949,나이키 x 톰 삭스 마스야드 슈 3.0 스페이스 캠프,IF2885-100,5251,25/09/19,339000,Natural/Sport Red/Maple/Sail,1100,1855128.0
7,501037,나이키 보메로 18 블랙 다크 스모크 그레이,HM6803-005,3948,NaN,179000,Black/Dark Smoke Grey/Light Smoke Grey/Black,1850,133740.0
8,611602,나이키 x 자크뮈스 문 슈 SP 오프 누아르 캐시미어,HV8547-001,13000,25/09/29,239000,Off Noir/Cashmere/Gum Light Brown/Neptune Gree...,2000,405878.0
9,416242,나이키 샥스 R4 화이트 레이서 블루,HQ1988-100,11000,NaN,179000,White/Racer Blue/Iron Grey/Metallic Silver,2000,140183.0


In [ ]:
# 1. 크롬 드라이버 셋팅
print("🌐 크롬 브라우저를 셋팅합니다...")
driver_path = ChromeDriverManager().install()
os.system(f"xattr -cr '{driver_path}'")
os.system("xattr -cr ~/.wdm")

options = webdriver.ChromeOptions()
options.add_argument("user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
options.add_experimental_option("detach", True)
driver = webdriver.Chrome(service=Service(driver_path), options=options)

today_date = datetime.now().date()

# 수집할 상품 ID 리스트 (남녀공용 나이키 신발 위주로 구성)
my_product_ids = [
    266659,
    501037,
    754815,
    526264,
    402232
]

ml_dataset2 = []

print("🚀 KREAM 남녀공용 타겟 데이터 수집\n" + "="*50)

for pid in my_product_ids:
    # 1. 기본 정보 
    basic_info = get_kream_basic_info(pid)
    
    # 2. 상세 정보
    details_info = get_kream_details(pid)
    
    # 3. 거래 내역 (최대 2000개 수집 및 대상 사이즈 평균)
    tx_data = get_kream_transactions(pid)
        
    # 4. 행(Row)으로 합치기 (최종 CSV 포맷 요구사항 반영)
    row_data = {
        "Product_ID": pid,
        "Name": basic_info["한글명"],
        "Model_No": details_info["모델번호"],
        "Wish_Count": basic_info["관심수"],
        "Release_Date": details_info["발매일"],
        "Retail_Price": details_info["발매가"],
        "Color": details_info.get("색상"),
        "Scraped_Tx_Count": tx_data["scraped_count"],
        "Golden_Avg_Price": tx_data["golden_avg_price"]
    }
    
    ml_dataset2.append(row_data)
    print(f"🎉 [{pid}] 수집 및 분석 완료! (긁어온 거래량: {tx_data['scraped_count']}개)\n" + "-"*50)

# 최종 데이터프레임 생성 및 CSV 저장
df_final2 = pd.DataFrame(ml_dataset2)
df_final2.to_csv("KREAM_Unisex_TargetSize_Dataset2.csv", index=False, encoding="utf-8-sig")

print("\n🎊 모든 데이터셋 생성이 완료되었습니다!")
print("폴더에 'KREAM_Unisex_TargetSize_Dataset2.csv' 파일이 생성되었습니다.")
display(df_final2)

🌐 크롬 브라우저를 셋팅합니다...
🚀 KREAM 남녀공용 타겟 데이터 수집

🚨 [266659] '혜택 더보기' 위의 '>' 버튼을 눌러 상세 정보를 펴주세요.


In [10]:
!pip install pytrends

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 24.7 MB/s eta 0:00:00 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pytrends]


In [11]:
import numpy as np
from pytrends.request import TrendReq

In [13]:
# 모델명 + 모델번호 그대로 -> 0점이 너무 많아서 X
# ------------------------------------------------------------------------------

print("🌐 구글 트렌드(제품명 & 모델번호 평균 검색) 추출을 시작합니다...")

# 1. 크림 크롤링 완료된 데이터 불러오기
df = pd.read_csv("KREAM_Unisex_TargetSize_Dataset.csv")

# 결과를 담을 리스트 준비
trend_w4_list, trend_w3_list, trend_w2_list, trend_w1_list = [], [], [], []
trend_dday_list = []
search_keywords_used = []

pytrends = TrendReq(hl='ko-KR', tz=540)

for index, row in df.iterrows():
    # 🌟 복잡한 정제 없이 크림 제품명과 모델번호를 그대로 가져옵니다.
    full_name = str(row['Name'])
    model_no = str(row['Model_No'])
    release_date_str = row['Release_Date']
    
    # 발매일이 없으면 구글 트렌드 '기준점'을 잡을 수 없으므로 스킵
    if pd.isna(release_date_str):
        print(f"⚠️ [{row.get('Product_ID', index)}] 발매일 정보가 없어 트렌드 분석을 스킵합니다.")
        for lst in [trend_w4_list, trend_w3_list, trend_w2_list, trend_w1_list, trend_dday_list]:
            lst.append(None)
        search_keywords_used.append(None)
        continue
        
    try:
        # 키워드 리스트 구성 (제품명 + 모델번호)
        kw_list = []
        if full_name != 'nan' and full_name.strip() != "":
            kw_list.append(full_name[:100]) # 구글 API 100자 에러 방지
        
        if model_no != 'nan' and model_no.strip() != "" and model_no != "None":
            kw_list.append(model_no.strip()[:100])
            
        search_keywords_used.append(", ".join(kw_list))
        
        # 날짜 파싱 (KREAM의 "24/05/12" 형식을 YYYY-MM-DD로 변환)
        try:
            if "/" in str(release_date_str) and len(str(release_date_str).split("/")[0]) == 2:
                release_date = pd.to_datetime("20" + str(release_date_str), format="%Y/%m/%d")
            else:
                release_date = pd.to_datetime(str(release_date_str))
        except:
            print(f"⚠️ [{row.get('Product_ID', index)}] 발매일 형식 오류: {release_date_str}")
            for lst in [trend_w4_list, trend_w3_list, trend_w2_list, trend_w1_list, trend_dday_list]:
                lst.append(None)
            continue

        # 발매일 기준 과거 28일(4주) ~ 발매일 당일까지 (총 29일치 데이터 요청)
        start_date = release_date - timedelta(days=28)
        timeframe = f"{start_date.strftime('%Y-%m-%d')} {release_date.strftime('%Y-%m-%d')}"
        
        print(f"\n👟 검색어: {kw_list} | 기간: {start_date.date()} ~ {release_date.date()}")
        
        # 구글 트렌드 동시 검색 요청
        pytrends.build_payload(kw_list=kw_list, timeframe=timeframe, geo='KR')
        df_trends = pytrends.interest_over_time()
        
        if df_trends.empty:
            print("   -> 검색량 데이터가 없습니다 (모두 0점)")
            for lst in [trend_w4_list, trend_w3_list, trend_w2_list, trend_w1_list, trend_dday_list]:
                lst.append(0)
        else:
            if 'isPartial' in df_trends.columns:
                df_trends = df_trends.drop(columns=['isPartial'])
            
            # 🌟 핵심: 이름 검색량과 모델번호 검색량의 '평균(mean)'을 일자별로 계산!
            # axis=1 은 같은 날짜의 데이터(가로줄)끼리 평균을 낸다는 뜻입니다.
            daily_avg_scores = df_trends.mean(axis=1).values
            
            # 뒤에서부터 슬라이싱하여 완벽하게 타임라인 분할 (D-Day 및 주차별 평균)
            t_dday = int(daily_avg_scores[-1]) if len(daily_avg_scores) >= 1 else 0                               # 당일 (D-Day)
            t_w1 = round(np.mean(daily_avg_scores[-8:-1]), 1) if len(daily_avg_scores) >= 8 else 0                # 1주전 (D-7 ~ D-1)
            t_w2 = round(np.mean(daily_avg_scores[-15:-8]), 1) if len(daily_avg_scores) >= 15 else 0              # 2주전 (D-14 ~ D-8)
            t_w3 = round(np.mean(daily_avg_scores[-22:-15]), 1) if len(daily_avg_scores) >= 22 else 0             # 3주전 (D-21 ~ D-15)
            t_w4 = round(np.mean(daily_avg_scores[-29:-22]), 1) if len(daily_avg_scores) >= 29 else 0             # 4주전 (D-28 ~ D-22)
            
            print(f"   🔥 [D-Day]: {t_dday} | [1주전]: {t_w1} | [2주전]: {t_w2} | [3주전]: {t_w3} | [4주전]: {t_w4}")
            
            trend_dday_list.append(t_dday)
            trend_w1_list.append(t_w1)
            trend_w2_list.append(t_w2)
            trend_w3_list.append(t_w3)
            trend_w4_list.append(t_w4)
            
        # 구글 서버 차단 방지를 위해 2초 휴식
        time.sleep(2)
        
    except Exception as e:
        print(f"   ❌ 에러 발생: {e}")
        for lst in [trend_w4_list, trend_w3_list, trend_w2_list, trend_w1_list, trend_dday_list]:
            lst.append(None)
        search_keywords_used.append(None)

# 2. 기존 데이터프레임에 파생 변수 6개 추가
df['Search_Keywords'] = search_keywords_used
df['Trend_W4_Avg'] = trend_w4_list
df['Trend_W3_Avg'] = trend_w3_list
df['Trend_W2_Avg'] = trend_w2_list
df['Trend_W1_Avg'] = trend_w1_list
df['Trend_D_Day'] = trend_dday_list

# 3. 최종 완성본 파일 저장!
df.to_csv("KREAM_Final_ML_Dataset.csv", index=False, encoding="utf-8-sig")

print("\n🎊 구글 트렌드 (제품명+모델번호 평균 산출) 결합 완료!")
display(df[['Name', 'Search_Keywords', 'Trend_W4_Avg', 'Trend_W3_Avg', 'Trend_W2_Avg', 'Trend_W1_Avg', 'Trend_D_Day']])

🌐 구글 트렌드(제품명 & 모델번호 평균 검색) 추출을 시작합니다...

👟 검색어: ['나이키 줌 보메로 5 SP 블랙', 'BV1358-003'] | 기간: 2023-06-05 ~ 2023-07-03
   -> 검색량 데이터가 없습니다 (모두 0점)

👟 검색어: ['나이키 ACG 루퍼스 라임스톤 앤 블랙', 'FV2923-200'] | 기간: 2024-04-12 ~ 2024-05-10
   -> 검색량 데이터가 없습니다 (모두 0점)

👟 검색어: ['나이키 에어포스 1 로우 고어텍스 비브람 오프 누아르 앤 블랙', 'HV5953-001'] | 기간: 2025-10-16 ~ 2025-11-13
   -> 검색량 데이터가 없습니다 (모두 0점)

👟 검색어: ['나이키 리액트X 리주버네이트 트리플 블랙', 'HV5060-001'] | 기간: 2025-01-01 ~ 2025-01-29
   -> 검색량 데이터가 없습니다 (모두 0점)

👟 검색어: ['나이키 x 니고 에어포스 3 로우 미드나잇 네이비 쉐도우 그레이', 'HV5032-400'] | 기간: 2026-01-09 ~ 2026-02-06
   -> 검색량 데이터가 없습니다 (모두 0점)
⚠️ [266659] 발매일 정보가 없어 트렌드 분석을 스킵합니다.

👟 검색어: ['나이키 x 톰 삭스 마스야드 슈 3.0 스페이스 캠프', 'IF2885-100'] | 기간: 2025-08-22 ~ 2025-09-19
   -> 검색량 데이터가 없습니다 (모두 0점)
⚠️ [501037] 발매일 정보가 없어 트렌드 분석을 스킵합니다.

👟 검색어: ['나이키 x 자크뮈스 문 슈 SP 오프 누아르 캐시미어', 'HV8547-001'] | 기간: 2025-09-01 ~ 2025-09-29
   -> 검색량 데이터가 없습니다 (모두 0점)
⚠️ [416242] 발매일 정보가 없어 트렌드 분석을 스킵합니다.

👟 검색어: ['나이키 자 3 EP 코발트 블리스 메탈릭 실버', 'HF2794-400'] | 기간: 2026-0

,Name,Search_Keywords,Trend_W4_Avg,Trend_W3_Avg,Trend_W2_Avg,Trend_W1_Avg,Trend_D_Day
0,나이키 줌 보메로 5 SP 블랙,"나이키 줌 보메로 5 SP 블랙, BV1358-003",0.0,0.0,0.0,0.0,0.0
1,나이키 ACG 루퍼스 라임스톤 앤 블랙,"나이키 ACG 루퍼스 라임스톤 앤 블랙, FV2923-200",0.0,0.0,0.0,0.0,0.0
2,나이키 에어포스 1 로우 고어텍스 비브람 오프 누아르 앤 블랙,"나이키 에어포스 1 로우 고어텍스 비브람 오프 누아르 앤 블랙, HV5953-001",0.0,0.0,0.0,0.0,0.0
3,나이키 리액트X 리주버네이트 트리플 블랙,"나이키 리액트X 리주버네이트 트리플 블랙, HV5060-001",0.0,0.0,0.0,0.0,0.0
4,나이키 x 니고 에어포스 3 로우 미드나잇 네이비 쉐도우 그레이,"나이키 x 니고 에어포스 3 로우 미드나잇 네이비 쉐도우 그레이, HV5032-400",0.0,0.0,0.0,0.0,0.0
5,나이키 V2K 런 블랙 앤트러사이트,NaN,NaN,NaN,NaN,NaN,NaN
6,나이키 x 톰 삭스 마스야드 슈 3.0 스페이스 캠프,"나이키 x 톰 삭스 마스야드 슈 3.0 스페이스 캠프, IF2885-100",0.0,0.0,0.0,0.0,0.0
7,나이키 보메로 18 블랙 다크 스모크 그레이,NaN,NaN,NaN,NaN,NaN,NaN
8,나이키 x 자크뮈스 문 슈 SP 오프 누아르 캐시미어,"나이키 x 자크뮈스 문 슈 SP 오프 누아르 캐시미어, HV8547-001",0.0,0.0,0.0,0.0,0.0
9,나이키 샥스 R4 화이트 레이서 블루,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
# 모델명(3단어만 나오게) + 모델번호 그대로 -> 조금 나아졌지만 별로 영향 없고 모델번호가 점수를 너무 많이 까먹음
# ------------------------------------------------------------------------------

print("🌐 구글 트렌드(초간단 3단어 정제 + 모델번호 평균) 추출을 시작합니다...")

# 1. 크림 크롤링 완료된 데이터 불러오기
df = pd.read_csv("KREAM_Unisex_TargetSize_Dataset.csv")

# 결과를 담을 리스트 준비
trend_w4_list, trend_w3_list, trend_w2_list, trend_w1_list = [], [], [], []
trend_dday_list = []
search_keywords_used = []

pytrends = TrendReq(hl='ko-KR', tz=540)

for index, row in df.iterrows():
    raw_name = str(row['Name'])
    model_no = str(row['Model_No'])
    release_date_str = row['Release_Date']
    
    if pd.isna(release_date_str):
        print(f"⚠️ [{row.get('Product_ID', index)}] 발매일 정보가 없어 트렌드 분석을 스킵합니다.")
        for lst in [trend_w4_list, trend_w3_list, trend_w2_list, trend_w1_list, trend_dday_list]:
            lst.append(None)
        search_keywords_used.append(None)
        continue
        
    try:
        # ==========================================
        # 🌟 핵심: 초간단 마법의 전처리 (딱 2줄!)
        # 1. (W), (GS) 같은 괄호 내용 삭제
        clean_name = re.sub(r'\(.*?\)', '', raw_name)
        # 2. 띄어쓰기 기준으로 딱 앞의 '3단어'까지만 남기기
        clean_name = " ".join(clean_name.split()[:3])
        # ==========================================

        kw_list = [clean_name]
        
        if model_no != 'nan' and model_no.strip() != "" and model_no != "None":
            kw_list.append(model_no.strip()[:100])
            
        search_keywords_used.append(", ".join(kw_list))
        
        # 날짜 파싱 
        try:
            if "/" in str(release_date_str) and len(str(release_date_str).split("/")[0]) == 2:
                release_date = pd.to_datetime("20" + str(release_date_str), format="%Y/%m/%d")
            else:
                release_date = pd.to_datetime(str(release_date_str))
        except:
            for lst in [trend_w4_list, trend_w3_list, trend_w2_list, trend_w1_list, trend_dday_list]:
                lst.append(None)
            continue

        # 발매일 기준 과거 28일(4주) ~ 발매일 당일까지 (총 29일치 데이터 요청)
        start_date = release_date - timedelta(days=28)
        timeframe = f"{start_date.strftime('%Y-%m-%d')} {release_date.strftime('%Y-%m-%d')}"
        
        print(f"\n👟 검색어: {kw_list} | 기간: {start_date.date()} ~ {release_date.date()}")
        
        # 구글 트렌드 동시 검색 요청
        pytrends.build_payload(kw_list=kw_list, timeframe=timeframe, geo='KR')
        df_trends = pytrends.interest_over_time()
        
        if df_trends.empty:
            print("   -> 검색량 데이터가 없습니다 (모두 0점)")
            for lst in [trend_w4_list, trend_w3_list, trend_w2_list, trend_w1_list, trend_dday_list]:
                lst.append(0)
        else:
            if 'isPartial' in df_trends.columns:
                df_trends = df_trends.drop(columns=['isPartial'])
            
            # 이름 검색량과 모델번호 검색량의 '평균(mean)'을 일자별로 계산
            daily_avg_scores = df_trends.mean(axis=1).values
            
            t_dday = int(daily_avg_scores[-1]) if len(daily_avg_scores) >= 1 else 0                              
            t_w1 = round(np.mean(daily_avg_scores[-8:-1]), 1) if len(daily_avg_scores) >= 8 else 0                
            t_w2 = round(np.mean(daily_avg_scores[-15:-8]), 1) if len(daily_avg_scores) >= 15 else 0              
            t_w3 = round(np.mean(daily_avg_scores[-22:-15]), 1) if len(daily_avg_scores) >= 22 else 0             
            t_w4 = round(np.mean(daily_avg_scores[-29:-22]), 1) if len(daily_avg_scores) >= 29 else 0             
            
            print(f"   🔥 [D-Day]: {t_dday} | [1주전]: {t_w1} | [2주전]: {t_w2} | [3주전]: {t_w3} | [4주전]: {t_w4}")
            
            trend_dday_list.append(t_dday)
            trend_w1_list.append(t_w1)
            trend_w2_list.append(t_w2)
            trend_w3_list.append(t_w3)
            trend_w4_list.append(t_w4)
            
        # 구글 서버 차단 방지를 위해 2초 휴식
        time.sleep(2)
        
    except Exception as e:
        print(f"   ❌ 에러 발생: {e}")
        for lst in [trend_w4_list, trend_w3_list, trend_w2_list, trend_w1_list, trend_dday_list]:
            lst.append(None)
        if len(search_keywords_used) < len(trend_w4_list):
            search_keywords_used.append(None)

# 2. 파생 변수 추가
df['Search_Keywords'] = search_keywords_used
df['Trend_W4_Avg'] = trend_w4_list
df['Trend_W3_Avg'] = trend_w3_list
df['Trend_W2_Avg'] = trend_w2_list
df['Trend_W1_Avg'] = trend_w1_list
df['Trend_D_Day'] = trend_dday_list

# 3. 최종 저장
df.to_csv("KREAM_Final_ML_Dataset.csv", index=False, encoding="utf-8-sig")

print("\n🎊 구글 트렌드 (3단어 정제 버전) 결합 완료!")
display(df[['Name', 'Search_Keywords', 'Trend_D_Day']])

🌐 구글 트렌드(초간단 3단어 정제 + 모델번호 평균) 추출을 시작합니다...

👟 검색어: ['나이키 줌 보메로', 'BV1358-003'] | 기간: 2023-06-05 ~ 2023-07-03
   -> 검색량 데이터가 없습니다 (모두 0점)

👟 검색어: ['나이키 ACG 루퍼스', 'FV2923-200'] | 기간: 2024-04-12 ~ 2024-05-10
   🔥 [D-Day]: 0 | [1주전]: 7.1 | [2주전]: 0.0 | [3주전]: 0.0 | [4주전]: 0.0

👟 검색어: ['나이키 에어포스 1', 'HV5953-001'] | 기간: 2025-10-16 ~ 2025-11-13
   -> 검색량 데이터가 없습니다 (모두 0점)

👟 검색어: ['나이키 리액트X 리주버네이트', 'HV5060-001'] | 기간: 2025-01-01 ~ 2025-01-29
   -> 검색량 데이터가 없습니다 (모두 0점)

👟 검색어: ['나이키 x 니고', 'HV5032-400'] | 기간: 2026-01-09 ~ 2026-02-06
   -> 검색량 데이터가 없습니다 (모두 0점)
⚠️ [266659] 발매일 정보가 없어 트렌드 분석을 스킵합니다.

👟 검색어: ['나이키 x 톰', 'IF2885-100'] | 기간: 2025-08-22 ~ 2025-09-19
   -> 검색량 데이터가 없습니다 (모두 0점)
⚠️ [501037] 발매일 정보가 없어 트렌드 분석을 스킵합니다.

👟 검색어: ['나이키 x 자크뮈스', 'HV8547-001'] | 기간: 2025-09-01 ~ 2025-09-29
   -> 검색량 데이터가 없습니다 (모두 0점)
⚠️ [416242] 발매일 정보가 없어 트렌드 분석을 스킵합니다.

👟 검색어: ['나이키 자 3', 'HF2794-400'] | 기간: 2026-01-16 ~ 2026-02-13
   -> 검색량 데이터가 없습니다 (모두 0점)

👟 검색어: ['나이키 ACG 파사드', 'HM7133-002'] | 기간: 2

,Name,Search_Keywords,Trend_D_Day
0,나이키 줌 보메로 5 SP 블랙,"나이키 줌 보메로, BV1358-003",0.0
1,나이키 ACG 루퍼스 라임스톤 앤 블랙,"나이키 ACG 루퍼스, FV2923-200",0.0
2,나이키 에어포스 1 로우 고어텍스 비브람 오프 누아르 앤 블랙,"나이키 에어포스 1, HV5953-001",0.0
3,나이키 리액트X 리주버네이트 트리플 블랙,"나이키 리액트X 리주버네이트, HV5060-001",0.0
4,나이키 x 니고 에어포스 3 로우 미드나잇 네이비 쉐도우 그레이,"나이키 x 니고, HV5032-400",0.0
5,나이키 V2K 런 블랙 앤트러사이트,NaN,NaN
6,나이키 x 톰 삭스 마스야드 슈 3.0 스페이스 캠프,"나이키 x 톰, IF2885-100",0.0
7,나이키 보메로 18 블랙 다크 스모크 그레이,NaN,NaN
8,나이키 x 자크뮈스 문 슈 SP 오프 누아르 캐시미어,"나이키 x 자크뮈스, HV8547-001",0.0
9,나이키 샥스 R4 화이트 레이서 블루,NaN,NaN


In [15]:
# 나이키 + 모델번호 -> 모델번호 버려야할듯
# ------------------------------------------------------------------------------

print("🌐 구글 트렌드(무조건 '나이키' + 모델번호 검색) 추출을 시작합니다...")

# 1. 크림 크롤링 완료된 데이터 불러오기
df = pd.read_csv("KREAM_Unisex_TargetSize_Dataset.csv")

# 결과를 담을 리스트 준비
trend_w4_list, trend_w3_list, trend_w2_list, trend_w1_list = [], [], [], []
trend_dday_list = []
search_keywords_used = []

pytrends = TrendReq(hl='ko-KR', tz=540)

for index, row in df.iterrows():
    model_no = str(row['Model_No'])
    release_date_str = row['Release_Date']
    
    if pd.isna(release_date_str):
        print(f"⚠️ [{row.get('Product_ID', index)}] 발매일 정보가 없어 트렌드 분석을 스킵합니다.")
        for lst in [trend_w4_list, trend_w3_list, trend_w2_list, trend_w1_list, trend_dday_list]:
            lst.append(None)
        search_keywords_used.append(None)
        continue
        
    try:
        # ==========================================
        # 🌟 핵심: 이름 다 무시하고 무조건 "나이키"로 고정!
        # ==========================================
        kw_list = ["나이키"]
        
        # 모델번호가 존재하면 같이 검색 리스트에 추가
        if model_no != 'nan' and model_no.strip() != "" and model_no != "None":
            kw_list.append(model_no.strip()[:100])
            
        search_keywords_used.append(", ".join(kw_list))
        
        # 날짜 파싱 
        try:
            if "/" in str(release_date_str) and len(str(release_date_str).split("/")[0]) == 2:
                release_date = pd.to_datetime("20" + str(release_date_str), format="%Y/%m/%d")
            else:
                release_date = pd.to_datetime(str(release_date_str))
        except:
            for lst in [trend_w4_list, trend_w3_list, trend_w2_list, trend_w1_list, trend_dday_list]:
                lst.append(None)
            continue

        # 발매일 기준 과거 28일(4주) ~ 발매일 당일까지 (총 29일치 데이터 요청)
        start_date = release_date - timedelta(days=28)
        timeframe = f"{start_date.strftime('%Y-%m-%d')} {release_date.strftime('%Y-%m-%d')}"
        
        print(f"\n👟 검색어: {kw_list} | 기간: {start_date.date()} ~ {release_date.date()}")
        
        # 구글 트렌드 동시 검색 요청
        pytrends.build_payload(kw_list=kw_list, timeframe=timeframe, geo='KR')
        df_trends = pytrends.interest_over_time()
        
        if df_trends.empty:
            print("   -> 검색량 데이터가 없습니다 (모두 0점)")
            for lst in [trend_w4_list, trend_w3_list, trend_w2_list, trend_w1_list, trend_dday_list]:
                lst.append(0)
        else:
            if 'isPartial' in df_trends.columns:
                df_trends = df_trends.drop(columns=['isPartial'])
            
            # '나이키'와 '모델번호' 검색량의 '평균(mean)'을 일자별로 계산
            daily_avg_scores = df_trends.mean(axis=1).values
            
            t_dday = int(daily_avg_scores[-1]) if len(daily_avg_scores) >= 1 else 0                              
            t_w1 = round(np.mean(daily_avg_scores[-8:-1]), 1) if len(daily_avg_scores) >= 8 else 0                
            t_w2 = round(np.mean(daily_avg_scores[-15:-8]), 1) if len(daily_avg_scores) >= 15 else 0              
            t_w3 = round(np.mean(daily_avg_scores[-22:-15]), 1) if len(daily_avg_scores) >= 22 else 0             
            t_w4 = round(np.mean(daily_avg_scores[-29:-22]), 1) if len(daily_avg_scores) >= 29 else 0             
            
            print(f"   🔥 [D-Day]: {t_dday} | [1주전]: {t_w1} | [2주전]: {t_w2} | [3주전]: {t_w3} | [4주전]: {t_w4}")
            
            trend_dday_list.append(t_dday)
            trend_w1_list.append(t_w1)
            trend_w2_list.append(t_w2)
            trend_w3_list.append(t_w3)
            trend_w4_list.append(t_w4)
            
        # 구글 서버 차단 방지를 위해 2초 휴식
        time.sleep(2)
        
    except Exception as e:
        print(f"   ❌ 에러 발생: {e}")
        for lst in [trend_w4_list, trend_w3_list, trend_w2_list, trend_w1_list, trend_dday_list]:
            lst.append(None)
        if len(search_keywords_used) < len(trend_w4_list):
            search_keywords_used.append(None)

# 2. 파생 변수 추가
df['Search_Keywords'] = search_keywords_used
df['Trend_W4_Avg'] = trend_w4_list
df['Trend_W3_Avg'] = trend_w3_list
df['Trend_W2_Avg'] = trend_w2_list
df['Trend_W1_Avg'] = trend_w1_list
df['Trend_D_Day'] = trend_dday_list

# 3. 최종 저장
df.to_csv("KREAM_Final_ML_Dataset.csv", index=False, encoding="utf-8-sig")

print("\n🎊 구글 트렌드 (무조건 '나이키' 버전) 결합 완료!")
display(df[['Name', 'Search_Keywords', 'Trend_D_Day']])

🌐 구글 트렌드(무조건 '나이키' + 모델번호 검색) 추출을 시작합니다...

👟 검색어: ['나이키', 'BV1358-003'] | 기간: 2023-06-05 ~ 2023-07-03
   🔥 [D-Day]: 37 | [1주전]: 39.2 | [2주전]: 40.5 | [3주전]: 43.4 | [4주전]: 38.0

👟 검색어: ['나이키', 'FV2923-200'] | 기간: 2024-04-12 ~ 2024-05-10
   🔥 [D-Day]: 19 | [1주전]: 23.6 | [2주전]: 29.1 | [3주전]: 21.5 | [4주전]: 22.6

👟 검색어: ['나이키', 'HV5953-001'] | 기간: 2025-10-16 ~ 2025-11-13
   🔥 [D-Day]: 27 | [1주전]: 27.7 | [2주전]: 21.9 | [3주전]: 21.9 | [4주전]: 22.2

👟 검색어: ['나이키', 'HV5060-001'] | 기간: 2025-01-01 ~ 2025-01-29
   🔥 [D-Day]: 50 | [1주전]: 38.9 | [2주전]: 36.5 | [3주전]: 40.1 | [4주전]: 38.2

👟 검색어: ['나이키', 'HV5032-400'] | 기간: 2026-01-09 ~ 2026-02-06
   🔥 [D-Day]: 31 | [1주전]: 35.5 | [2주전]: 38.7 | [3주전]: 38.1 | [4주전]: 35.4
⚠️ [266659] 발매일 정보가 없어 트렌드 분석을 스킵합니다.

👟 검색어: ['나이키', 'IF2885-100'] | 기간: 2025-08-22 ~ 2025-09-19
   🔥 [D-Day]: 38 | [1주전]: 42.5 | [2주전]: 41.3 | [3주전]: 40.8 | [4주전]: 40.1
⚠️ [501037] 발매일 정보가 없어 트렌드 분석을 스킵합니다.

👟 검색어: ['나이키', 'HV8547-001'] | 기간: 2025-09-01 ~ 2025-09-29
   🔥 [D-Day]: 46 | [1주전

,Name,Search_Keywords,Trend_D_Day
0,나이키 줌 보메로 5 SP 블랙,"나이키, BV1358-003",37.0
1,나이키 ACG 루퍼스 라임스톤 앤 블랙,"나이키, FV2923-200",19.0
2,나이키 에어포스 1 로우 고어텍스 비브람 오프 누아르 앤 블랙,"나이키, HV5953-001",27.0
3,나이키 리액트X 리주버네이트 트리플 블랙,"나이키, HV5060-001",50.0
4,나이키 x 니고 에어포스 3 로우 미드나잇 네이비 쉐도우 그레이,"나이키, HV5032-400",31.0
5,나이키 V2K 런 블랙 앤트러사이트,NaN,NaN
6,나이키 x 톰 삭스 마스야드 슈 3.0 스페이스 캠프,"나이키, IF2885-100",38.0
7,나이키 보메로 18 블랙 다크 스모크 그레이,NaN,NaN
8,나이키 x 자크뮈스 문 슈 SP 오프 누아르 캐시미어,"나이키, HV8547-001",46.0
9,나이키 샥스 R4 화이트 레이서 블루,NaN,NaN


In [16]:
print("🌐 구글 트렌드(오직 '나이키' 단일 검색) 추출을 시작합니다...")

# 1. 크림 크롤링 완료된 데이터 불러오기
df = pd.read_csv("KREAM_Unisex_TargetSize_Dataset.csv")

# 결과를 담을 리스트 준비
trend_w4_list, trend_w3_list, trend_w2_list, trend_w1_list = [], [], [], []
trend_dday_list = []

pytrends = TrendReq(hl='ko-KR', tz=540)

for index, row in df.iterrows():
    release_date_str = row['Release_Date']
    
    if pd.isna(release_date_str):
        print(f"⚠️ [{row.get('Product_ID', index)}] 발매일 정보가 없어 트렌드 분석을 스킵합니다.")
        for lst in [trend_w4_list, trend_w3_list, trend_w2_list, trend_w1_list, trend_dday_list]:
            lst.append(None)
        continue
        
    try:
        # ==========================================
        # 🌟 핵심: 모델번호 다 빼고 오직 "나이키" 하나로만 고정!
        # ==========================================
        kw_list = ["나이키"]
        
        # 날짜 파싱 
        try:
            if "/" in str(release_date_str) and len(str(release_date_str).split("/")[0]) == 2:
                release_date = pd.to_datetime("20" + str(release_date_str), format="%Y/%m/%d")
            else:
                release_date = pd.to_datetime(str(release_date_str))
        except:
            for lst in [trend_w4_list, trend_w3_list, trend_w2_list, trend_w1_list, trend_dday_list]:
                lst.append(None)
            continue

        # 발매일 기준 과거 28일(4주) ~ 발매일 당일까지 (총 29일치 데이터 요청)
        start_date = release_date - timedelta(days=28)
        timeframe = f"{start_date.strftime('%Y-%m-%d')} {release_date.strftime('%Y-%m-%d')}"
        
        print(f"\n👟 검색어: {kw_list} | 기간: {start_date.date()} ~ {release_date.date()}")
        
        # 구글 트렌드 단일 검색 요청
        pytrends.build_payload(kw_list=kw_list, timeframe=timeframe, geo='KR')
        df_trends = pytrends.interest_over_time()
        
        if df_trends.empty:
            print("   -> 검색량 데이터가 없습니다 (0점)")
            for lst in [trend_w4_list, trend_w3_list, trend_w2_list, trend_w1_list, trend_dday_list]:
                lst.append(0)
        else:
            if 'isPartial' in df_trends.columns:
                df_trends = df_trends.drop(columns=['isPartial'])
            
            # 오직 '나이키'의 일자별 점수 가져오기
            daily_scores = df_trends.iloc[:, 0].values
            
            t_dday = int(daily_scores[-1]) if len(daily_scores) >= 1 else 0                              
            t_w1 = round(np.mean(daily_scores[-8:-1]), 1) if len(daily_scores) >= 8 else 0                
            t_w2 = round(np.mean(daily_scores[-15:-8]), 1) if len(daily_scores) >= 15 else 0              
            t_w3 = round(np.mean(daily_scores[-22:-15]), 1) if len(daily_scores) >= 22 else 0             
            t_w4 = round(np.mean(daily_scores[-29:-22]), 1) if len(daily_scores) >= 29 else 0             
            
            print(f"   🔥 [D-Day]: {t_dday} | [1주전]: {t_w1} | [2주전]: {t_w2} | [3주전]: {t_w3} | [4주전]: {t_w4}")
            
            trend_dday_list.append(t_dday)
            trend_w1_list.append(t_w1)
            trend_w2_list.append(t_w2)
            trend_w3_list.append(t_w3)
            trend_w4_list.append(t_w4)
            
        # 구글 서버 차단 방지를 위해 2초 휴식
        time.sleep(2)
        
    except Exception as e:
        print(f"   ❌ 에러 발생: {e}")
        for lst in [trend_w4_list, trend_w3_list, trend_w2_list, trend_w1_list, trend_dday_list]:
            lst.append(None)

# 2. 파생 변수 추가 (검색어는 항상 "나이키"로 고정)
df['Search_Keyword'] = "나이키"
df['Trend_W4_Avg'] = trend_w4_list
df['Trend_W3_Avg'] = trend_w3_list
df['Trend_W2_Avg'] = trend_w2_list
df['Trend_W1_Avg'] = trend_w1_list
df['Trend_D_Day'] = trend_dday_list

# 3. 최종 저장
df.to_csv("KREAM_Final_ML_Dataset.csv", index=False, encoding="utf-8-sig")

print("\n🎊 구글 트렌드 (오직 '나이키' 단일 검색 버전) 결합 완료!")
display(df[['Name', 'Search_Keyword', 'Trend_D_Day', 'Trend_W1_Avg']])

🌐 구글 트렌드(오직 '나이키' 단일 검색) 추출을 시작합니다...

👟 검색어: ['나이키'] | 기간: 2023-06-05 ~ 2023-07-03
   🔥 [D-Day]: 74 | [1주전]: 78.4 | [2주전]: 81.0 | [3주전]: 86.9 | [4주전]: 76.0

👟 검색어: ['나이키'] | 기간: 2024-04-12 ~ 2024-05-10
   🔥 [D-Day]: 38 | [1주전]: 47.1 | [2주전]: 58.3 | [3주전]: 43.0 | [4주전]: 45.1

👟 검색어: ['나이키'] | 기간: 2025-10-16 ~ 2025-11-13
   🔥 [D-Day]: 55 | [1주전]: 55.4 | [2주전]: 43.7 | [3주전]: 43.9 | [4주전]: 44.4

👟 검색어: ['나이키'] | 기간: 2025-01-01 ~ 2025-01-29
   🔥 [D-Day]: 100 | [1주전]: 77.7 | [2주전]: 73.0 | [3주전]: 80.3 | [4주전]: 76.4

👟 검색어: ['나이키'] | 기간: 2026-01-09 ~ 2026-02-06
   🔥 [D-Day]: 63 | [1주전]: 71.0 | [2주전]: 77.4 | [3주전]: 76.1 | [4주전]: 70.9
⚠️ [266659] 발매일 정보가 없어 트렌드 분석을 스킵합니다.

👟 검색어: ['나이키'] | 기간: 2025-08-22 ~ 2025-09-19
   🔥 [D-Day]: 76 | [1주전]: 85.0 | [2주전]: 82.6 | [3주전]: 81.6 | [4주전]: 80.1
⚠️ [501037] 발매일 정보가 없어 트렌드 분석을 스킵합니다.

👟 검색어: ['나이키'] | 기간: 2025-09-01 ~ 2025-09-29
   🔥 [D-Day]: 92 | [1주전]: 82.9 | [2주전]: 83.6 | [3주전]: 83.9 | [4주전]: 84.1
⚠️ [416242] 발매일 정보가 없어 트렌드 분석을 스킵합니다.

👟 검색어: ['나이키'

,Name,Search_Keyword,Trend_D_Day,Trend_W1_Avg
0,나이키 줌 보메로 5 SP 블랙,나이키,74.0,78.4
1,나이키 ACG 루퍼스 라임스톤 앤 블랙,나이키,38.0,47.1
2,나이키 에어포스 1 로우 고어텍스 비브람 오프 누아르 앤 블랙,나이키,55.0,55.4
3,나이키 리액트X 리주버네이트 트리플 블랙,나이키,100.0,77.7
4,나이키 x 니고 에어포스 3 로우 미드나잇 네이비 쉐도우 그레이,나이키,63.0,71.0
5,나이키 V2K 런 블랙 앤트러사이트,나이키,NaN,NaN
6,나이키 x 톰 삭스 마스야드 슈 3.0 스페이스 캠프,나이키,76.0,85.0
7,나이키 보메로 18 블랙 다크 스모크 그레이,나이키,NaN,NaN
8,나이키 x 자크뮈스 문 슈 SP 오프 누아르 캐시미어,나이키,92.0,82.9
9,나이키 샥스 R4 화이트 레이서 블루,나이키,NaN,NaN


In [17]:
print("🌐 구글 트렌드('나이키' + 제품명 3단어 듀얼 검색) 추출을 시작합니다...")

# 1. 크림 크롤링 완료된 데이터 불러오기
df = pd.read_csv("KREAM_Unisex_TargetSize_Dataset.csv")

# 결과를 담을 리스트 준비
trend_w4_list, trend_w3_list, trend_w2_list, trend_w1_list = [], [], [], []
trend_dday_list = []
search_keywords_used = []

pytrends = TrendReq(hl='ko-KR', tz=540)

for index, row in df.iterrows():
    raw_name = str(row['Name'])
    release_date_str = row['Release_Date']
    
    if pd.isna(release_date_str):
        print(f"⚠️ [{row.get('Product_ID', index)}] 발매일 정보가 없어 트렌드 분석을 스킵합니다.")
        for lst in [trend_w4_list, trend_w3_list, trend_w2_list, trend_w1_list, trend_dday_list]:
            lst.append(None)
        search_keywords_used.append(None)
        continue
        
    try:
        # ==========================================
        # 🌟 핵심 1: 제품명 앞 3단어만 예쁘게 깎아내기
        # ==========================================
        clean_name = re.sub(r'\(.*?\)', '', raw_name)
        clean_name = clean_name.replace(' x ', ' ')
        clean_name = re.sub(r'\b(SP|SE|OG|GTX|High|Low|Mid)\b', '', clean_name, flags=re.IGNORECASE)
        clean_name = re.sub(r'\s+', ' ', clean_name).strip()
        
        clean_name_3words = " ".join(clean_name.split()[:3])
        
        # ==========================================
        # 🌟 핵심 2: 검색어 리스트 세팅 ['나이키', '정제된 3단어']
        # ==========================================
        kw_list = ["나이키"]
        if clean_name_3words:
            kw_list.append(clean_name_3words)
            
        search_keywords_used.append(", ".join(kw_list))
        
        # 날짜 파싱 
        try:
            if "/" in str(release_date_str) and len(str(release_date_str).split("/")[0]) == 2:
                release_date = pd.to_datetime("20" + str(release_date_str), format="%Y/%m/%d")
            else:
                release_date = pd.to_datetime(str(release_date_str))
        except:
            for lst in [trend_w4_list, trend_w3_list, trend_w2_list, trend_w1_list, trend_dday_list]:
                lst.append(None)
            continue

        # 발매일 기준 과거 28일(4주) ~ 발매일 당일까지 (총 29일치 데이터 요청)
        start_date = release_date - timedelta(days=28)
        timeframe = f"{start_date.strftime('%Y-%m-%d')} {release_date.strftime('%Y-%m-%d')}"
        
        print(f"\n👟 검색어: {kw_list} | 기간: {start_date.date()} ~ {release_date.date()}")
        
        # 구글 트렌드 동시 검색 요청
        pytrends.build_payload(kw_list=kw_list, timeframe=timeframe, geo='KR')
        df_trends = pytrends.interest_over_time()
        
        if df_trends.empty:
            print("   -> 검색량 데이터가 없습니다 (모두 0점)")
            for lst in [trend_w4_list, trend_w3_list, trend_w2_list, trend_w1_list, trend_dday_list]:
                lst.append(0)
        else:
            if 'isPartial' in df_trends.columns:
                df_trends = df_trends.drop(columns=['isPartial'])
            
            # 🌟 [나이키] 점수와 [3단어] 점수의 '평균(mean)'을 일자별로 계산!
            daily_avg_scores = df_trends.mean(axis=1).values
            
            t_dday = int(daily_avg_scores[-1]) if len(daily_avg_scores) >= 1 else 0                              
            t_w1 = round(np.mean(daily_avg_scores[-8:-1]), 1) if len(daily_avg_scores) >= 8 else 0                
            t_w2 = round(np.mean(daily_avg_scores[-15:-8]), 1) if len(daily_avg_scores) >= 15 else 0              
            t_w3 = round(np.mean(daily_avg_scores[-22:-15]), 1) if len(daily_avg_scores) >= 22 else 0             
            t_w4 = round(np.mean(daily_avg_scores[-29:-22]), 1) if len(daily_avg_scores) >= 29 else 0             
            
            print(f"   🔥 [D-Day]: {t_dday} | [1주전]: {t_w1} | [2주전]: {t_w2} | [3주전]: {t_w3} | [4주전]: {t_w4}")
            
            trend_dday_list.append(t_dday)
            trend_w1_list.append(t_w1)
            trend_w2_list.append(t_w2)
            trend_w3_list.append(t_w3)
            trend_w4_list.append(t_w4)
            
        # 구글 서버 차단 방지를 위해 2초 휴식
        time.sleep(2)
        
    except Exception as e:
        print(f"   ❌ 에러 발생: {e}")
        for lst in [trend_w4_list, trend_w3_list, trend_w2_list, trend_w1_list, trend_dday_list]:
            lst.append(None)
        if len(search_keywords_used) < len(trend_w4_list):
            search_keywords_used.append(None)

# 2. 파생 변수 추가
df['Search_Keywords'] = search_keywords_used
df['Trend_W4_Avg'] = trend_w4_list
df['Trend_W3_Avg'] = trend_w3_list
df['Trend_W2_Avg'] = trend_w2_list
df['Trend_W1_Avg'] = trend_w1_list
df['Trend_D_Day'] = trend_dday_list

# 3. 최종 저장
df.to_csv("KREAM_Final_ML_Dataset.csv", index=False, encoding="utf-8-sig")

print("\n🎊 구글 트렌드 ('나이키' + 제품명 3단어 평균) 결합 완료!")
display(df[['Name', 'Search_Keywords', 'Trend_D_Day', 'Trend_W1_Avg']])

🌐 구글 트렌드('나이키' + 제품명 3단어 듀얼 검색) 추출을 시작합니다...

👟 검색어: ['나이키', '나이키 줌 보메로'] | 기간: 2023-06-05 ~ 2023-07-03
   🔥 [D-Day]: 37 | [1주전]: 39.2 | [2주전]: 40.5 | [3주전]: 43.4 | [4주전]: 38.0

👟 검색어: ['나이키', '나이키 ACG 루퍼스'] | 기간: 2024-04-12 ~ 2024-05-10
   🔥 [D-Day]: 19 | [1주전]: 23.8 | [2주전]: 29.1 | [3주전]: 21.5 | [4주전]: 22.6

👟 검색어: ['나이키', '나이키 에어포스 1'] | 기간: 2025-10-16 ~ 2025-11-13
   🔥 [D-Day]: 27 | [1주전]: 27.7 | [2주전]: 21.9 | [3주전]: 21.9 | [4주전]: 22.2

👟 검색어: ['나이키', '나이키 리액트X 리주버네이트'] | 기간: 2025-01-01 ~ 2025-01-29
   🔥 [D-Day]: 50 | [1주전]: 38.9 | [2주전]: 36.5 | [3주전]: 40.1 | [4주전]: 38.2

👟 검색어: ['나이키', '나이키 니고 에어포스'] | 기간: 2026-01-09 ~ 2026-02-06
   🔥 [D-Day]: 31 | [1주전]: 35.5 | [2주전]: 38.7 | [3주전]: 38.1 | [4주전]: 35.4
⚠️ [266659] 발매일 정보가 없어 트렌드 분석을 스킵합니다.

👟 검색어: ['나이키', '나이키 톰 삭스'] | 기간: 2025-08-22 ~ 2025-09-19
   🔥 [D-Day]: 38 | [1주전]: 42.5 | [2주전]: 41.3 | [3주전]: 40.8 | [4주전]: 40.1
⚠️ [501037] 발매일 정보가 없어 트렌드 분석을 스킵합니다.

👟 검색어: ['나이키', '나이키 자크뮈스 문'] | 기간: 2025-09-01 ~ 2025-09-29
   🔥 [D-Day]: 46 

,Name,Search_Keywords,Trend_D_Day,Trend_W1_Avg
0,나이키 줌 보메로 5 SP 블랙,"나이키, 나이키 줌 보메로",37.0,39.2
1,나이키 ACG 루퍼스 라임스톤 앤 블랙,"나이키, 나이키 ACG 루퍼스",19.0,23.8
2,나이키 에어포스 1 로우 고어텍스 비브람 오프 누아르 앤 블랙,"나이키, 나이키 에어포스 1",27.0,27.7
3,나이키 리액트X 리주버네이트 트리플 블랙,"나이키, 나이키 리액트X 리주버네이트",50.0,38.9
4,나이키 x 니고 에어포스 3 로우 미드나잇 네이비 쉐도우 그레이,"나이키, 나이키 니고 에어포스",31.0,35.5
5,나이키 V2K 런 블랙 앤트러사이트,NaN,NaN,NaN
6,나이키 x 톰 삭스 마스야드 슈 3.0 스페이스 캠프,"나이키, 나이키 톰 삭스",38.0,42.5
7,나이키 보메로 18 블랙 다크 스모크 그레이,NaN,NaN,NaN
8,나이키 x 자크뮈스 문 슈 SP 오프 누아르 캐시미어,"나이키, 나이키 자크뮈스 문",46.0,41.4
9,나이키 샥스 R4 화이트 레이서 블루,NaN,NaN,NaN
